In [0]:
%pip install faker polars

In [0]:
import random
import uuid
import json
from datetime import datetime, timedelta
import polars as pl
import time
from faker import Faker

In [0]:
from config import CUSTOMER_IDS, PRODUCT_IDS, CHANNELS, PAYMENT_METHODS, ORDER_STATUSES, DELIVERY_STATUSES, CATEGORIES, FLAT_CATEGORIES, N_CUSTOMERS, N_PRODUCTS, N_ORDERS, N_EVENTS   

In [0]:
fake = Faker("es_MX")  # Spanish locale for more variety, mix with en_US
fake_en = Faker("en_US")

In [0]:
random.seed(42)
OUTPUT_DIR  = "/Volumes/e2e_databricks/raw/storage/events"
BATCH_SIZE  = 20    # events per file — change this to control throughput
SLEEP_SECS  = 10    # seconds between batches

In [0]:
def generate_events_stream(batch_size: int = BATCH_SIZE) -> pl.DataFrame:
    """
    Generates one micro-batch of events and writes it as a unique JSONL file.
    Each call produces a NEW file — Auto Loader picks them up incrementally.

    Dirty patterns:
    - ~8%  out-of-order events
    - ~6%  duplicate events (mobile retry)
    - Inconsistent JSON schema (v1 vs v2)
    - ~4%  null session_id
    - Rotating device_id for same user
    """
    import os
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    EVENT_TYPES    = ["page_view", "page_view", "page_view", "add_to_cart", "add_to_cart", "purchase", "return_request"]
    rotating_users = random.sample(CUSTOMER_IDS, int(N_CUSTOMERS * 0.10))
    device_map     = {uid: [str(uuid.uuid4()), str(uuid.uuid4())] for uid in rotating_users}

    rows      = []
    base_time = datetime.now() - timedelta(seconds=batch_size * 2)

    for i in range(batch_size):
        cid        = random.choice(CUSTOMER_IDS)
        event_type = random.choice(EVENT_TYPES)
        event_time = base_time + timedelta(seconds=i * 2 + random.uniform(-5, 5))
        session_id = str(uuid.uuid4()) if random.random() > 0.04 else None
        pid        = random.choice(PRODUCT_IDS) if event_type != "page_view" else None
        device_id  = random.choice(device_map[cid]) if cid in device_map else f"DEV-{cid:06d}"

        app_version = random.choice(["v1", "v1", "v2"])
        if app_version == "v1":
            payload = {"ts": event_time.isoformat(), "user": cid, "event": event_type,
                       "product": pid, "device": device_id, "session": session_id, "app_ver": "v1"}
        else:
            payload = {"event_timestamp": event_time.isoformat(), "customer_id": cid,
                       "event_name": event_type, "product_id": pid, "device_id": device_id,
                       "session_id": session_id, "app_version": "v2",
                       "platform": random.choice(["ios", "android", "web"])}

        rows.append({
            "event_id":    str(uuid.uuid4()),
            "customer_id": cid,
            "event_type":  event_type,
            "product_id":  pid,
            "device_id":   device_id,
            "session_id":  session_id,
            "event_time":  event_time.isoformat(),
            "payload_json": json.dumps(payload),
            "app_version": app_version,
            "channel":     random.choice(CHANNELS),
        })

    df = pl.DataFrame(rows)

    # inject ~6% duplicates
    n_dupes = max(1, int(batch_size * 0.06))
    df      = pl.concat([df, df.sample(n_dupes, with_replacement=True)])

    # inject ~8% out-of-order timestamps
    n_ooo       = max(1, int(len(df) * 0.08))
    shuffle_idx = random.sample(range(len(df)), n_ooo)
    times       = df["event_time"].to_list()
    shuffled    = [times[i] for i in shuffle_idx]
    random.shuffle(shuffled)
    for i, idx in enumerate(shuffle_idx):
        times[idx] = shuffled[i]
    df = df.with_columns(pl.Series("event_time", times)).sort("event_time")

    # unique filename per batch — Auto Loader detects each new file
    ts       = datetime.now().strftime("%Y%m%d_%H%M%S")
    uid      = str(uuid.uuid4())[:8]
    filepath = f"{OUTPUT_DIR}/events_{ts}_{uid}.jsonl"

    with open(filepath, "w") as f:
        for row in df.to_dicts():
            f.write(json.dumps(row) + "\n")

    print(f"✓ {filepath.split('/')[-1]} — {len(df):,} events  ({n_dupes} dupes)")
    return df

In [0]:
batch_num = 0
while True:
    batch_num += 1
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Batch {batch_num}")
    generate_events_stream(BATCH_SIZE)
    time.sleep(SLEEP_SECS)
